# Day 5 — RAG Foundations

In this module, we implement a Retrieval-Augmented Generation (RAG) system. We cover how to connect LLMs to external private databases, extract relevant facts using vector databases (like ChromaDB), and stuff retrieved content into prompts to ensure models reply with grounded data.

## Objectives
- Build a basic cosine similarity text query matcher in pure Python.
- Set up and store document vectors in ChromaDB.
- Implement prompt context injection with Groq.
- **Extended Implementation:** Build a recursive character text splitter with custom overlaps, implement metadata-filtered querying, and write a similarity score threshold gate.

In [ ]:
# Install dependencies
!pip install chromadb groq numpy python-dotenv -q
print("✅ Libraries installed.")

## 1. Cosine Similarity Search from Scratch
We implement a basic bag-of-words text vectorization and cosine similarity matcher in Python to inspect the math behind semantic database matching.

In [ ]:
import math
import re
from collections import Counter

CORPUS = [
    "A Machine Learning Engineer must be fluent in Python and PyTorch.",
    "MLOps skills include Docker, CI/CD, and model monitoring.",
    "Vector databases power retrieval for RAG systems.",
    "Frontend engineers focus on React and TypeScript."
]

def tokenize(text: str):
    return re.findall(r"[a-z0-9]+", text.lower())

def embed(text: str) -> Counter:
    return Counter(tokenize(text))

def cosine_similarity(a: Counter, b: Counter) -> float:
    common = set(a) & set(b)
    dot = sum(a[t] * b[t] for t in common)
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return dot / (na * nb) if na and nb else 0.0

# Search
query = "Tell me about vector databases"
q_vec = embed(query)
for doc in CORPUS:
    score = cosine_similarity(q_vec, embed(doc))
    print(f"Similarity: {score:.3f} | '{doc}'")

## 2. ChromaDB Vector Database
We use ChromaDB to automate embedding generation and run vector index matches on input documents.

In [ ]:
import chromadb

# Initialize database in-memory
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="roles_collection")

# Add documents
collection.add(
    documents=CORPUS,
    ids=[f"id_{i}" for i in range(len(CORPUS))]
)

# Query
results = collection.query(
    query_texts=["vector database and RAG"],
    n_results=1
)
print("Query Results:")
print(results['documents'][0])

## 3. Ingestion and Generation Loop
We fetch query matches from ChromaDB, inject the matched texts into prompt template strings, and ask the Groq model to complete answers.

In [ ]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def rag_pipeline(user_query):
    # 1. Retrieve
    results = collection.query(query_texts=[user_query], n_results=1)
    retrieved_chunk = results['documents'][0][0]
    
    # 2. Stuff Prompt
    system_prompt = (
        "You are an assistant. Answer the user's question USING ONLY the provided context. "
        "If the context doesn't contain the answer, say 'I don't know.'\n\n"
        f"[CONTEXT]\n{retrieved_chunk}"
    )
    
    # 3. Generate
    completion = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ],
        temperature=0.0
    )
    return completion.choices[0].message.content

print("RAG Answer:")
print(rag_pipeline("What skills does an MLOps engineer need?"))

---
# Extended Implementation
Below are implementation techniques covering custom text splitters, metadata filtering, and threshold validation.

### 1. Recursive Character Text Splitter from scratch
We implement a manual sliding text splitter that chunks documents into smaller strings with a defined overlap parameter to preserve context boundaries.

In [ ]:
def custom_text_splitter(text: str, chunk_size: int, overlap: int) -> List[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += (chunk_size - overlap)
        if end >= len(text):
            break
    return chunks

doc_text = "CareerForge is an advanced platform containing career mapping, curriculum alignment, and multi-agent systems."
split_chunks = custom_text_splitter(doc_text, chunk_size=30, overlap=10)

print(f"Document split into {len(split_chunks)} chunks:")
for idx, chunk in enumerate(split_chunks):
    print(f"  Chunk {idx}: '{chunk}' (Length: {len(chunk)})")

### 2. Semantic + Metadata Filter Querying
We query ChromaDB using semantic queries combined with strict metadata dictionary constraints.

In [ ]:
meta_collection = chroma_client.create_collection(name="meta_collection")

# Add chunks with metadata tags
meta_collection.add(
    documents=CORPUS,
    metadatas=[
        {"domain": "engineering"},
        {"domain": "operations"},
        {"domain": "infrastructure"},
        {"domain": "design"}
    ],
    ids=[f"meta_id_{i}" for i in range(len(CORPUS))]
)

# Query with domain = 'engineering'
filtered_results = meta_collection.query(
    query_texts=["skills and frameworks"],
    where={"domain": "engineering"},
    n_results=1
)

print("Filtered Query Result:")
print("Documents:", filtered_results['documents'][0])
print("Metadata:", filtered_results['metadatas'][0])

### 3. Similarity Score Threshold Guard
We implement a threshold check on distance results. If distance values exceed limits, the system raises an exception or reports that no relevant context was found.

In [ ]:
def query_with_threshold(query_str, threshold=1.0):
    res = collection.query(query_texts=[query_str], n_results=1)
    distance = res['distances'][0][0]
    doc = res['documents'][0][0]
    
    print(f"Query: '{query_str}' | Retrieved Distance: {distance:.4f}")
    
    if distance > threshold:
        print("Distance exceeds similarity threshold. Refusing to feed LLM context.")
        return "I'm sorry, I couldn't find any relevant facts in the database to answer that question."
        
    return f"Context Match: {doc}"

print(query_with_threshold("Explain vector database systems.", threshold=1.0))
print("-----------------------------------")
print(query_with_threshold("How do you bake a strawberry cheesecake?", threshold=1.0))